# Run BKT experiments on ASSISTments 2017

## Imports

In [ ]:
!pip -q install git+https://github.com/CAHLR/pyBKT.git pandas numpy scikit-learn

from google.colab import drive
drive.mount("/content/drive")

import os
import time
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss
from pyBKT.models import Model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_DIR = "/content/drive/MyDrive/education-ml-research"

print("Setup complete.")
print("Project directory exists:", os.path.isdir(BASE_DIR))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive
Setup complete.
Project directory exists: True


## Inspect datasets

In [ ]:
DATA_PATH_17 = os.path.join(
    BASE_DIR, "ASSISTments2017",
    "anonymized_full_release_competition_dataset.csv"
)
GROUPS_PATH_17 = os.path.join(
    BASE_DIR, "ASSISTments2017",
    "student_irt_ability_groups.csv"
)

assist17_raw = pd.read_csv(DATA_PATH_17)
irt_groups_17 = pd.read_csv(GROUPS_PATH_17)

print("Dataset shape:", assist17_raw.shape)
print("Dataset columns:")
print(assist17_raw.columns.tolist())

print("\nIRT groups shape:", irt_groups_17.shape)
print("IRT group columns:")
print(irt_groups_17.columns.tolist())

print("\nDataset preview:")
display(assist17_raw.head())

print("\nIRT groups preview:")
display(irt_groups_17.head())

/tmp/ipykernel_1220/4243771876.py:10: DtypeWarning: Columns (76,77) have mixed types. Specify dtype option on import or set low_memory=False.
  assist17_raw = pd.read_csv(DATA_PATH_17)


Dataset shape: (942816, 82)
Dataset columns:
['studentId', 'MiddleSchoolId', 'InferredGender', 'SY ASSISTments Usage', 'AveKnow', 'AveCarelessness', 'AveCorrect', 'NumActions', 'AveResBored', 'AveResEngcon', 'AveResConf', 'AveResFrust', 'AveResOfftask', 'AveResGaming', 'action_num', 'skill', 'problemId', 'problemType', 'assignmentId', 'assistmentId', 'startTime', 'endTime', 'timeTaken', 'correct', 'original', 'hint', 'hintCount', 'hintTotal', 'scaffold', 'bottomHint', 'attemptCount', 'frIsHelpRequest', 'frPast5HelpRequest', 'frPast8HelpRequest', 'stlHintUsed', 'past8BottomOut', 'totalFrPercentPastWrong', 'totalFrPastWrongCount', 'frPast5WrongCount', 'frPast8WrongCount', 'totalFrTimeOnSkill', 'timeSinceSkill', 'frWorkingInSchool', 'totalFrAttempted', 'totalFrSkillOpportunities', 'responseIsFillIn', 'responseIsChosen', 'endsWithScaffolding', 'endsWithAutoScaffolding', 'frTimeTakenOnScaffolding', 'frTotalSkillOpportunitiesScaffolding', 'totalFrSkillOpportunitiesByScaffolding', 'frIsHelpRe

,studentId,MiddleSchoolId,InferredGender,SY ASSISTments Usage,AveKnow,AveCarelessness,AveCorrect,NumActions,AveResBored,AveResEngcon,...,RES_CONFUSED,RES_FRUSTRATED,RES_OFFTASK,RES_GAMING,Ln-1,Ln,MCAS,Enrolled,Selective,isSTEM
0,8,2,Male,2004-2005,0.35242,0.18328,0.48390,1056,0.20839,0.67913,...,0.00000,0.00000,0.78559,0.00026,0.13000,0.06119,45,0,0,NaN
1,8,2,Male,2004-2005,0.35242,0.18328,0.48390,1056,0.20839,0.67913,...,0.88745,0.00000,0.46825,0.00148,0.06119,0.21351,45,0,0,NaN
2,8,2,Male,2004-2005,0.35242,0.18328,0.48390,1056,0.20839,0.67913,...,0.88745,0.00000,0.46825,0.00148,0.11600,0.03331,45,0,0,NaN
3,8,2,Male,2004-2005,0.35242,0.18328,0.48390,1056,0.20839,0.67913,...,0.00000,0.00000,0.10842,0.01066,0.11600,0.03331,45,0,0,NaN
4,8,2,Male,2004-2005,0.35242,0.18328,0.48390,1056,0.20839,0.67913,...,0.00000,0.00000,0.10842,0.01066,0.03331,0.11839,45,0,0,NaN



IRT groups preview:


,studentId,theta_all,theta_first,theta_quartile,theta_sd_group
0,8,0.14541,0.65945,Q4,0 to +0.5 SD
1,9,0.04003,0.33471,Q3,0 to +0.5 SD
2,11,-0.62656,-0.53872,Q2,-1.5 to -1.0 SD
3,25,0.33921,0.54421,Q4,+0.5 to +1.0 SD
4,27,-0.26449,0.36049,Q3,-0.5 to 0 SD


In [ ]:
print("Unique students in dataset:", assist17_raw["studentId"].nunique())
print("Unique students with IRT estimates:", irt_groups_17["studentId"].nunique())

print("\nMissing values in key columns:")
display(
    assist17_raw[
        ["studentId", "skill", "problemId", "correct", "action_num"]
    ].isna().sum().to_frame("missing_count")
)

print("\nCorrectness values:")
print(assist17_raw["correct"].value_counts(dropna=False).head(10))

print("\nUnique skills:", assist17_raw["skill"].nunique())
print("Unique problems:", assist17_raw["problemId"].nunique())

print("\nDuplicate student IDs in IRT file:",
      irt_groups_17["studentId"].duplicated().sum())

print("\nIRT quartile counts:")
print(irt_groups_17["theta_quartile"].value_counts(dropna=False).sort_index())

print("\nIRT SD-group counts:")
print(irt_groups_17["theta_sd_group"].value_counts(dropna=False))

Unique students in dataset: 1709
Unique students with IRT estimates: 1709

Missing values in key columns:


,missing_count
studentId,0
skill,0
problemId,0
correct,0
action_num,0



Correctness values:
correct
0    591446
1    351370
Name: count, dtype: int64

Unique skills: 102
Unique problems: 3162

Duplicate student IDs in IRT file: 0

IRT quartile counts:
theta_quartile
Q1    428
Q2    427
Q3    427
Q4    427
Name: count, dtype: int64

IRT SD-group counts:
theta_sd_group
-1.0 to -0.5 SD    407
-1.5 to -1.0 SD    330
-0.5 to 0 SD       314
0 to +0.5 SD       232
< -1.5 SD          136
+0.5 to +1.0 SD    134
+1.0 to +1.5 SD     82
> +1.5 SD           74
Name: count, dtype: int64


## Prepare BKT data

In [ ]:
# Prepare chronological interaction data
assist17 = assist17_raw[
    ["studentId", "skill", "problemId", "correct", "action_num"]
].copy()

assist17["studentId"] = assist17["studentId"].astype(str)
assist17["skill"] = assist17["skill"].astype(str)
assist17["correct"] = pd.to_numeric(assist17["correct"], errors="coerce")
assist17["action_num"] = pd.to_numeric(
    assist17["action_num"], errors="coerce"
)

assist17 = assist17.dropna(
    subset=["studentId", "skill", "correct", "action_num"]
).copy()

assist17 = assist17[assist17["correct"].isin([0, 1])].copy()
assist17["correct"] = assist17["correct"].astype(int)

assist17 = assist17.sort_values(
    ["studentId", "action_num"]
).reset_index(drop=True)

# Prepare IRT group labels
irt_groups_17["studentId"] = irt_groups_17["studentId"].astype(str)

print("Cleaned interactions:", len(assist17))
print("Unique students:", assist17["studentId"].nunique())
print("Unique skills:", assist17["skill"].nunique())
print("Correct response rate:", round(assist17["correct"].mean(), 6))
print("Duplicate student-action pairs:",
      assist17.duplicated(["studentId", "action_num"]).sum())

Cleaned interactions: 942816
Unique students: 1709
Unique skills: 102
Correct response rate: 0.372681
Duplicate student-action pairs: 0


## Create the train/test split

In [ ]:
train_students_17 = []
test_students_17 = []

for quartile in ["Q1", "Q2", "Q3", "Q4"]:
    students = irt_groups_17.loc[
        irt_groups_17["theta_quartile"] == quartile,
        "studentId"
    ].to_numpy()

    train_ids, test_ids = train_test_split(
        students,
        test_size=0.20,
        random_state=SEED
    )

    train_students_17.extend(train_ids)
    test_students_17.extend(test_ids)

train_set_17 = set(train_students_17)
test_set_17 = set(test_students_17)

assist17_train = assist17[
    assist17["studentId"].isin(train_set_17)
].copy()

assist17_test = assist17[
    assist17["studentId"].isin(test_set_17)
].copy()

print("Training students:", len(train_set_17))
print("Test students:", len(test_set_17))
print("Student overlap:", len(train_set_17 & test_set_17))
print("Training interactions:", len(assist17_train))
print("Test interactions:", len(assist17_test))

print("\nTest students by IRT quartile:")
print(
    irt_groups_17[
        irt_groups_17["studentId"].isin(test_set_17)
    ]["theta_quartile"].value_counts().sort_index()
)

Training students: 1365
Test students: 344
Student overlap: 0
Training interactions: 760517
Test interactions: 182299

Test students by IRT quartile:
theta_quartile
Q1    86
Q2    86
Q3    86
Q4    86
Name: count, dtype: int64


## Format data for BKT

In [ ]:
def prepare_bkt_data_17(data):
    prepared = data[
        ["action_num", "studentId", "skill", "correct"]
    ].copy()

    prepared = prepared.rename(columns={
        "action_num": "order_id",
        "studentId": "user_id",
        "skill": "skill_id"
    })

    prepared["order_id"] = pd.to_numeric(
        prepared["order_id"], errors="coerce"
    )
    prepared["correct"] = pd.to_numeric(
        prepared["correct"], errors="coerce"
    )

    prepared = prepared.dropna(
        subset=["order_id", "user_id", "skill_id", "correct"]
    )
    prepared["skill_id"] = prepared["skill_id"].astype(str)
    prepared["user_id"] = prepared["user_id"].astype(str)
    prepared["correct"] = prepared["correct"].astype(int)

    return prepared.sort_values(
        ["user_id", "order_id"]
    ).reset_index(drop=True)


bkt_train_17 = prepare_bkt_data_17(assist17_train)
bkt_test_17 = prepare_bkt_data_17(assist17_test)

print("Training interactions:", len(bkt_train_17))
print("Test interactions:", len(bkt_test_17))
print("Training students:", bkt_train_17["user_id"].nunique())
print("Test students:", bkt_test_17["user_id"].nunique())
print("Training skills:", bkt_train_17["skill_id"].nunique())
print("Test skills:", bkt_test_17["skill_id"].nunique())
print("Train/test student overlap:", len(
    set(bkt_train_17["user_id"]) & set(bkt_test_17["user_id"])
))

Training interactions: 760517
Test interactions: 182299
Training students: 1365
Test students: 344
Training skills: 93
Test skills: 101
Train/test student overlap: 0


## Create a split where all skills are in training

In [ ]:
all_skills_17 = set(assist17["skill"].astype(str).unique())

split_candidates_17 = []

for seed in range(42, 143):
    train_ids = []
    test_ids = []

    for quartile in ["Q1", "Q2", "Q3", "Q4"]:
        students = irt_groups_17.loc[
            irt_groups_17["theta_quartile"] == quartile,
            "studentId"
        ].to_numpy()

        train_part, test_part = train_test_split(
            students,
            test_size=0.20,
            random_state=seed
        )

        train_ids.extend(train_part)
        test_ids.extend(test_part)

    train_set = set(train_ids)
    test_set = set(test_ids)

    train_data = assist17[assist17["studentId"].isin(train_set)]
    test_data = assist17[assist17["studentId"].isin(test_set)]

    train_skills = set(train_data["skill"].astype(str).unique())
    test_skills = set(test_data["skill"].astype(str).unique())

    split_candidates_17.append({
        "seed": seed,
        "train_students": len(train_set),
        "test_students": len(test_set),
        "train_skills": len(train_skills),
        "test_skills": len(test_skills),
        "unseen_test_skills": len(test_skills - train_skills),
        "train_interactions": len(train_data),
        "test_interactions": len(test_data)
    })

split_candidates_17 = pd.DataFrame(split_candidates_17)

display(
    split_candidates_17.sort_values(
        ["unseen_test_skills", "train_skills"],
        ascending=[True, False]
    ).head(15)
)

,seed,train_students,test_students,train_skills,test_skills,unseen_test_skills,train_interactions,test_interactions
1,43,1365,344,102,91,0,753200,189616
2,44,1365,344,102,92,0,754266,188550
3,45,1365,344,102,92,0,751172,191644
4,46,1365,344,102,92,0,745265,197551
6,48,1365,344,102,92,0,752124,190692
7,49,1365,344,102,92,0,742204,200612
9,51,1365,344,102,92,0,745460,197356
11,53,1365,344,102,91,0,756402,186414
13,55,1365,344,102,92,0,749224,193592
14,56,1365,344,102,92,0,755371,187445


## Select seed 43

In [ ]:
SEED_17 = 43

train_students_17 = []
test_students_17 = []

for quartile in ["Q1", "Q2", "Q3", "Q4"]:
    students = irt_groups_17.loc[
        irt_groups_17["theta_quartile"] == quartile,
        "studentId"
    ].to_numpy()

    train_ids, test_ids = train_test_split(
        students,
        test_size=0.20,
        random_state=SEED_17
    )

    train_students_17.extend(train_ids)
    test_students_17.extend(test_ids)

train_set_17 = set(map(str, train_students_17))
test_set_17 = set(map(str, test_students_17))

assist17_train = assist17[
    assist17["studentId"].isin(train_set_17)
].copy()

assist17_test = assist17[
    assist17["studentId"].isin(test_set_17)
].copy()

bkt_train_17 = prepare_bkt_data_17(assist17_train)
bkt_test_17 = prepare_bkt_data_17(assist17_test)

train_skills_17 = set(bkt_train_17["skill_id"].unique())
test_skills_17 = set(bkt_test_17["skill_id"].unique())

print("Seed:", SEED_17)
print("Training students:", len(train_set_17))
print("Test students:", len(test_set_17))
print("Student overlap:", len(train_set_17 & test_set_17))
print("Training interactions:", len(bkt_train_17))
print("Test interactions:", len(bkt_test_17))
print("Training skills:", len(train_skills_17))
print("Test skills:", len(test_skills_17))
print("Unseen test skills:", len(test_skills_17 - train_skills_17))

Seed: 43
Training students: 1365
Test students: 344
Student overlap: 0
Training interactions: 753200
Test interactions: 189616
Training skills: 102
Test skills: 91
Unseen test skills: 0


## Fit the model

In [ ]:
start = time.time()

bkt_model_17 = Model(
    seed=42,
    num_fits=1,
    parallel=True
)

bkt_model_17.fit(
    data=bkt_train_17,
    defaults={
        "order_id": "order_id",
        "student_id": "user_id",
        "skill_name": "skill_id",
        "correct": "correct"
    }
)

print(f"BKT training time: {time.time() - start:.2f} seconds")
print("BKT fitting complete.")
print("Skills fitted:", len(bkt_model_17.fit_model))

BKT training time: 8.53 seconds
BKT fitting complete.
Skills fitted: 102


## Predict on the test set

In [ ]:
bkt_predictions_17 = bkt_model_17.predict(data=bkt_test_17.copy())

print("Prediction rows:", len(bkt_predictions_17))
print("Test rows:", len(bkt_test_17))
print("Prediction columns:", bkt_predictions_17.columns.tolist())
print(
    "Missing predicted probabilities:",
    bkt_predictions_17["correct_predictions"].isna().sum()
)

Prediction rows: 189616
Test rows: 189616
Prediction columns: ['order_id', 'user_id', 'skill_id', 'correct', 'correct_predictions', 'state_predictions']
Missing predicted probabilities: 0


## Extract AUCs and Brier Score for training data as baseline

In [ ]:
bkt_train_predictions_17 = bkt_model_17.predict(data=bkt_train_17.copy())

y_train_true_17 = bkt_train_predictions_17["correct"].astype(int)
y_train_prob_17 = bkt_train_predictions_17["correct_predictions"].astype(float)

print("Training prediction rows:", len(bkt_train_predictions_17))
print("Training AUC:", round(
    roc_auc_score(y_train_true_17, y_train_prob_17), 6
))
print("Training Brier Score:", round(
    brier_score_loss(y_train_true_17, y_train_prob_17), 6
))

Training prediction rows: 753200
Training AUC: 0.633912
Training Brier Score: 0.220287


## Extract test data

In [ ]:
# Attach IRT ability groups to test predictions
bkt_results_17 = bkt_predictions_17.merge(
    irt_groups_17[
        ["studentId", "theta_all", "theta_quartile", "theta_sd_group"]
    ].rename(columns={"studentId": "user_id"}),
    on="user_id",
    how="left",
    validate="many_to_one"
)

def calculate_metrics_17(data, group_name=None, group_col=None):
    y_true = data["correct"].astype(int)
    y_prob = data["correct_predictions"].astype(float)

    return {
        "Ability Group": group_name if group_name is not None else "Overall",
        "Students": data["user_id"].nunique(),
        "Interactions": len(data),
        "AUC": roc_auc_score(y_true, y_prob) if y_true.nunique() == 2 else np.nan,
        "Brier Score": brier_score_loss(y_true, y_prob)
    }

overall_test_17 = pd.DataFrame([
    calculate_metrics_17(bkt_results_17)
])

quartile_order_17 = ["Q1", "Q2", "Q3", "Q4"]
quartile_rows_17 = []

for group in quartile_order_17:
    subset = bkt_results_17[
        bkt_results_17["theta_quartile"] == group
    ]
    quartile_rows_17.append(calculate_metrics_17(subset, group))

quartile_results_17 = pd.DataFrame(quartile_rows_17)

sd_order_17 = [
    "< -1.5 SD",
    "-1.5 to -1.0 SD",
    "-1.0 to -0.5 SD",
    "-0.5 to 0 SD",
    "0 to +0.5 SD",
    "+0.5 to +1.0 SD",
    "+1.0 to +1.5 SD",
    "> +1.5 SD"
]

sd_rows_17 = []

for group in sd_order_17:
    subset = bkt_results_17[
        bkt_results_17["theta_sd_group"] == group
    ]
    if not subset.empty:
        sd_rows_17.append(calculate_metrics_17(subset, group))

sd_results_17 = pd.DataFrame(sd_rows_17)

print("Missing ability groups:",
      bkt_results_17["theta_quartile"].isna().sum())

print("\nOverall test results:")
display(overall_test_17.round(6))

print("\nResults by IRT quartile:")
display(quartile_results_17.round(6))

print("\nResults by IRT standard-deviation group:")
display(sd_results_17.round(6))

Missing ability groups: 0

Overall test results:


,Ability Group,Students,Interactions,AUC,Brier Score
0,Overall,344,189616,0.63341,0.22148



Results by IRT quartile:


,Ability Group,Students,Interactions,AUC,Brier Score
0,Q1,86,53028,0.56142,0.20509
1,Q2,86,54062,0.60331,0.21640
2,Q3,86,46543,0.62052,0.23002
3,Q4,86,35983,0.65238,0.24221



Results by IRT standard-deviation group:


,Ability Group,Students,Interactions,AUC,Brier Score
0,< -1.5 SD,26,14065,0.53505,0.19710
1,-1.5 to -1.0 SD,67,43263,0.57060,0.20808
2,-1.0 to -0.5 SD,84,54080,0.60712,0.21755
3,-0.5 to 0 SD,64,33593,0.61740,0.22884
4,0 to +0.5 SD,44,20333,0.63670,0.23701
5,+0.5 to +1.0 SD,26,11435,0.62590,0.24643
6,+1.0 to +1.5 SD,14,5711,0.63533,0.24607
7,> +1.5 SD,19,7136,0.61944,0.24189


## BKT Results — ASSISTments 2017

### Overall Performance

| Metric | Score |
|---|---:|
| AUC | 0.63341 |
| Brier Score | 0.22148 |

### Performance by IRT Ability Quartile

| Ability Group | Students | Interactions | AUC | Brier Score |
|---|---:|---:|---:|---:|
| Q1 (Lowest θ) | 86 | 53,028 | 0.56142 | 0.20509 |
| Q2 | 86 | 54,062 | 0.60331 | 0.21640 |
| Q3 | 86 | 46,543 | 0.62052 | 0.23002 |
| Q4 (Highest θ) | 86 | 35,983 | 0.65238 | 0.24221 |

### Performance by IRT Standard-Deviation Group

| Ability Group | Students | Interactions | AUC | Brier Score |
|---|---:|---:|---:|---:|
| Below −1.5 SD | 26 | 14,065 | 0.53505 | 0.19710 |
| −1.5 to −1.0 SD | 67 | 43,263 | 0.57060 | 0.20808 |
| −1.0 to −0.5 SD | 84 | 54,080 | 0.60712 | 0.21755 |
| −0.5 to 0 SD | 64 | 33,593 | 0.61740 | 0.22884 |
| 0 to +0.5 SD | 44 | 20,333 | 0.63670 | 0.23701 |
| +0.5 to +1.0 SD | 26 | 11,435 | 0.62590 | 0.24643 |
| +1.0 to +1.5 SD | 14 | 5,711 | 0.63533 | 0.24607 |
| Above +1.5 SD | 19 | 7,136 | 0.61944 | 0.24189 |